In [1]:
# Import Required Libraries
import polars as pl
import numpy as np

In [9]:
pl.Config.set_tbl_cols(-1)       # display.max.columns -> None (-1 means all)
pl.Config.set_tbl_rows(-1)       # display.max.rows -> None (-1 means all)
pl.Config.set_float_precision(2) # display.precision -> 2

polars.config.Config

In [10]:
# Load the Hip Replacement CCG data
# Load the Knee Replacement Provider data
df = pl.read_parquet(f'./data/interim/2.0-Hip-preprocessing.parquet')


# Make a copy of the original dataframe
df_original = df.clone()
print("Original dataframe copied to df_original.")

Original dataframe copied to df_original.


In [11]:
# Display basic information about the dataset
print("Shape of the dataset:", df.shape)
print("\nColumn names:")
print(df.columns)
print("\nData types:")
print(df.schema)
print("\nFirst 5 rows:")
df.head()

Shape of the dataset: (124844, 82)

Column names:
['Provider Code', 'Procedure', 'Revision Flag', 'Year', 'Age Band', 'Gender', 'Pre-Op Q Assisted', 'Pre-Op Q Assisted By', 'Pre-Op Q Symptom Period', 'Pre-Op Q Previous Surgery', 'Pre-Op Q Living Arrangements', 'Pre-Op Q Disability', 'Heart Disease', 'High Bp', 'Stroke', 'Circulation', 'Lung Disease', 'Diabetes', 'Kidney Disease', 'Nervous System', 'Liver Disease', 'Cancer', 'Depression', 'Arthritis', 'Pre-Op Q Mobility', 'Pre-Op Q Self-Care', 'Pre-Op Q Activity', 'Pre-Op Q Discomfort', 'Pre-Op Q Anxiety', 'Pre-Op Q EQ5D Index Profile', 'Pre-Op Q EQ5D Index', 'Post-Op Q Assisted', 'Post-Op Q Assisted By', 'Post-Op Q Living Arrangements', 'Post-Op Q Disability', 'Post-Op Q Mobility', 'Post-Op Q Self-Care', 'Post-Op Q Activity', 'Post-Op Q Discomfort', 'Post-Op Q Anxiety', 'Post-Op Q Satisfaction', 'Post-Op Q Sucess', 'Post-Op Q Allergy', 'Post-Op Q Bleeding', 'Post-Op Q Wound', 'Post-Op Q Urine', 'Post-Op Q Further Surgery', 'Post-Op Q R

Provider Code,Procedure,Revision Flag,Year,Age Band,Gender,Pre-Op Q Assisted,Pre-Op Q Assisted By,Pre-Op Q Symptom Period,Pre-Op Q Previous Surgery,Pre-Op Q Living Arrangements,Pre-Op Q Disability,Heart Disease,High Bp,Stroke,Circulation,Lung Disease,Diabetes,Kidney Disease,Nervous System,Liver Disease,Cancer,Depression,Arthritis,Pre-Op Q Mobility,Pre-Op Q Self-Care,Pre-Op Q Activity,Pre-Op Q Discomfort,Pre-Op Q Anxiety,Pre-Op Q EQ5D Index Profile,Pre-Op Q EQ5D Index,Post-Op Q Assisted,Post-Op Q Assisted By,Post-Op Q Living Arrangements,Post-Op Q Disability,Post-Op Q Mobility,Post-Op Q Self-Care,Post-Op Q Activity,Post-Op Q Discomfort,Post-Op Q Anxiety,Post-Op Q Satisfaction,Post-Op Q Sucess,Post-Op Q Allergy,Post-Op Q Bleeding,Post-Op Q Wound,Post-Op Q Urine,Post-Op Q Further Surgery,Post-Op Q Readmitted,Post-Op Q EQ5D Index Profile,Post-Op Q EQ5D Index,Hip Replacement EQ5D Index Post-Op Q Predicted,Pre-Op Q EQ VAS,Post-Op Q EQ VAS,Hip Replacement EQ VAS Post-Op Q Predicted,Hip Replacement Pre-Op Q Pain,Hip Replacement Pre-Op Q Sudden Pain,Hip Replacement Pre-Op Q Night Pain,Hip Replacement Pre-Op Q Washing,Hip Replacement Pre-Op Q Transport,Hip Replacement Pre-Op Q Dressing,Hip Replacement Pre-Op Q Shopping,Hip Replacement Pre-Op Q Walking,Hip Replacement Pre-Op Q Limping,Hip Replacement Pre-Op Q Stairs,Hip Replacement Pre-Op Q Standing,Hip Replacement Pre-Op Q Work,Hip Replacement Pre-Op Q Score,Hip Replacement Post-Op Q Pain,Hip Replacement Post-Op Q Sudden Pain,Hip Replacement Post-Op Q Night Pain,Hip Replacement Post-Op Q Washing,Hip Replacement Post-Op Q Transport,Hip Replacement Post-Op Q Dressing,Hip Replacement Post-Op Q Shopping,Hip Replacement Post-Op Q Walking,Hip Replacement Post-Op Q Limping,Hip Replacement Post-Op Q Stairs,Hip Replacement Post-Op Q Standing,Hip Replacement Post-Op Q Work,Hip Replacement Post-Op Q Score,Hip Replacement OHS Post-Op Q Predicted,CSVYear
cat,cat,u8,cat,cat,f32,u8,i32,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u32,f32,u8,i32,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u32,f32,f32,u16,u16,cat,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,f32,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,f32,cat,cat
"""ADP02""","""Hip Replacement""",0,"""2016/17""",null,null,2,1,2,1,1,1,0,0,0,0,0,0,1,0,0,0,0,1,null,null,null,null,null,99999,null,2,0,2,2,1,1,2,1,1,2,1,2,2,2,2,2,2,11211,0.88,null,null,80,null,0,2,1,3,1,2,3,2,1,0,2,2,19.00,3,4,2,4,3,2,4,4,3,2,2,3,36.00,"""3.777.303.196""","""2016/17"""
"""ADP02""","""Hip Replacement""",0,"""2016/17""",null,null,2,1,3,2,1,1,0,0,0,0,0,0,0,0,0,0,0,0,2,2,2,3,1,22231,0.05,2,0,1,2,2,1,1,1,1,1,1,2,2,2,2,2,2,21111,0.85,0.67,30,70,"""6.592.244.892""",0,1,0,2,2,1,1,1,0,2,2,1,13.00,4,4,4,4,3,3,4,4,4,4,4,4,46.00,"""3.558.681.573""","""2016/17"""
"""ADP02""","""Hip Replacement""",0,"""2016/17""",null,null,1,1,2,2,1,1,0,1,0,0,0,0,0,0,0,0,0,1,2,2,3,3,2,22332,-0.07,2,0,1,1,2,1,2,1,1,2,1,2,2,2,2,2,2,21211,0.81,0.70,40,60,"""6.810.332.786""",1,0,0,1,1,1,0,1,0,1,1,0,7.00,1,4,3,3,2,3,2,4,1,2,4,2,31.00,"""3.293.405.433""","""2016/17"""
"""ADP02""","""Hip Replacement""",0,"""2016/17""",null,null,2,1,4,2,1,2,0,0,0,0,0,0,0,1,0,0,0,0,2,2,2,3,1,22231,0.05,2,0,1,2,2,1,2,2,2,3,2,2,2,2,1,2,1,21222,0.62,0.71,50,55,"""6.883.315.758""",0,4,0,2,2,1,3,2,1,2,2,2,21.00,0,0,0,3,2,2,2,1,1,2,2,0,15.00,"""3.867.962.866""","""2016/17"""
"""ADP02""","""Hip Replacement""",0,"""2016/17""",null,null,2,1,4,2,1,2,0,0,0,0,0,0,0,0,0,0,0,0,2,2,2,3,2,22232,-0.02,2,0,1,2,1,1,1,1,1,1,1,2,2,2,2,2,2,11111,1.00,0.74,20,90,"""7.226.409.643""",0,0,0,1,1,1,2,2,0,2,1,0,10.00,4,4,3,4,4,4,4,4,4,4,4,4,47.00,"""3.686.746.026""","""2016/17"""


In [12]:
# Returns a 1-row DataFrame with counts for each column
print(df.null_count())

shape: (1, 82)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ Pro ┆ Pro ┆ Rev ┆ Yea ┆ Age ┆ Gen ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Hea ┆ Hig ┆ Str ┆ Cir ┆ Lun ┆ Dia ┆ Kid ┆ Ner ┆ Liv ┆ Can ┆ Dep ┆ Art ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Hip ┆ Pre ┆ Pos ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ CSV

In [15]:
# Identify column types
numeric_dtypes = [pl.Int8, pl.Int16, pl.Int32, pl.Int64, pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64, pl.Float32, pl.Float64]
string_dtypes  = [pl.Utf8, pl.Categorical]  # parquet may store string columns as Categorical

numerical_cols = [c for c in df.columns if df[c].dtype in numeric_dtypes]
string_cols    = [c for c in df.columns if df[c].dtype in string_dtypes]

numerical_df = df.select(numerical_cols)
string_df    = df.select(string_cols)

print("List of numerical columns:")
print(numerical_cols)
print(f"\nList of string/categorical columns:")
print(string_cols)
print(f"\nNumerical dataframe shape: {numerical_df.shape}")
print(df.describe())

List of numerical columns:
['Revision Flag', 'Gender', 'Pre-Op Q Assisted', 'Pre-Op Q Assisted By', 'Pre-Op Q Symptom Period', 'Pre-Op Q Previous Surgery', 'Pre-Op Q Living Arrangements', 'Pre-Op Q Disability', 'Heart Disease', 'High Bp', 'Stroke', 'Circulation', 'Lung Disease', 'Diabetes', 'Kidney Disease', 'Nervous System', 'Liver Disease', 'Cancer', 'Depression', 'Arthritis', 'Pre-Op Q Mobility', 'Pre-Op Q Self-Care', 'Pre-Op Q Activity', 'Pre-Op Q Discomfort', 'Pre-Op Q Anxiety', 'Pre-Op Q EQ5D Index Profile', 'Pre-Op Q EQ5D Index', 'Post-Op Q Assisted', 'Post-Op Q Assisted By', 'Post-Op Q Living Arrangements', 'Post-Op Q Disability', 'Post-Op Q Mobility', 'Post-Op Q Self-Care', 'Post-Op Q Activity', 'Post-Op Q Discomfort', 'Post-Op Q Anxiety', 'Post-Op Q Satisfaction', 'Post-Op Q Sucess', 'Post-Op Q Allergy', 'Post-Op Q Bleeding', 'Post-Op Q Wound', 'Post-Op Q Urine', 'Post-Op Q Further Surgery', 'Post-Op Q Readmitted', 'Post-Op Q EQ5D Index Profile', 'Post-Op Q EQ5D Index', 'Hip 

In [16]:
# Print dataset overview
print("DATASET OVERVIEW:")
print(f"Total observations: {df.shape[0]}")
print(f"Total variables/columns: {df.shape[1]}")
print(f"\nNumerical variables/columns: {len(numerical_cols)}")
print(f"String variables/columns: {len(string_cols)}")

print(f"\nNumerical columns ({len(numerical_cols)}):")
print(numerical_cols)

print(f"\nString columns ({len(string_cols)}):")
print(string_cols)

print(f"\nNumerical dataframe shape: {numerical_df.shape}")
print(f"String dataframe shape: {string_df.shape}")

DATASET OVERVIEW:
Total observations: 124844
Total variables/columns: 82

Numerical variables/columns: 75
String variables/columns: 7

Numerical columns (75):
['Revision Flag', 'Gender', 'Pre-Op Q Assisted', 'Pre-Op Q Assisted By', 'Pre-Op Q Symptom Period', 'Pre-Op Q Previous Surgery', 'Pre-Op Q Living Arrangements', 'Pre-Op Q Disability', 'Heart Disease', 'High Bp', 'Stroke', 'Circulation', 'Lung Disease', 'Diabetes', 'Kidney Disease', 'Nervous System', 'Liver Disease', 'Cancer', 'Depression', 'Arthritis', 'Pre-Op Q Mobility', 'Pre-Op Q Self-Care', 'Pre-Op Q Activity', 'Pre-Op Q Discomfort', 'Pre-Op Q Anxiety', 'Pre-Op Q EQ5D Index Profile', 'Pre-Op Q EQ5D Index', 'Post-Op Q Assisted', 'Post-Op Q Assisted By', 'Post-Op Q Living Arrangements', 'Post-Op Q Disability', 'Post-Op Q Mobility', 'Post-Op Q Self-Care', 'Post-Op Q Activity', 'Post-Op Q Discomfort', 'Post-Op Q Anxiety', 'Post-Op Q Satisfaction', 'Post-Op Q Sucess', 'Post-Op Q Allergy', 'Post-Op Q Bleeding', 'Post-Op Q Wound', '

In [17]:
# Check for missing values
null_counts = df.null_count()
print("Missing values per column:")
print(null_counts)

print("\nPercentage of missing values:")
print(null_counts / df.shape[0] * 100)

print("\nFirst 10 rows:")
df.head(10)

Missing values per column:
shape: (1, 82)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ Pro ┆ Pro ┆ Rev ┆ Yea ┆ Age ┆ Gen ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Hea ┆ Hig ┆ Str ┆ Cir ┆ Lun ┆ Dia ┆ Kid ┆ Ner ┆ Liv ┆ Can ┆ Dep ┆ Art ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Hip ┆ Pre ┆ Pos ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ 

Provider Code,Procedure,Revision Flag,Year,Age Band,Gender,Pre-Op Q Assisted,Pre-Op Q Assisted By,Pre-Op Q Symptom Period,Pre-Op Q Previous Surgery,Pre-Op Q Living Arrangements,Pre-Op Q Disability,Heart Disease,High Bp,Stroke,Circulation,Lung Disease,Diabetes,Kidney Disease,Nervous System,Liver Disease,Cancer,Depression,Arthritis,Pre-Op Q Mobility,Pre-Op Q Self-Care,Pre-Op Q Activity,Pre-Op Q Discomfort,Pre-Op Q Anxiety,Pre-Op Q EQ5D Index Profile,Pre-Op Q EQ5D Index,Post-Op Q Assisted,Post-Op Q Assisted By,Post-Op Q Living Arrangements,Post-Op Q Disability,Post-Op Q Mobility,Post-Op Q Self-Care,Post-Op Q Activity,Post-Op Q Discomfort,Post-Op Q Anxiety,Post-Op Q Satisfaction,Post-Op Q Sucess,Post-Op Q Allergy,Post-Op Q Bleeding,Post-Op Q Wound,Post-Op Q Urine,Post-Op Q Further Surgery,Post-Op Q Readmitted,Post-Op Q EQ5D Index Profile,Post-Op Q EQ5D Index,Hip Replacement EQ5D Index Post-Op Q Predicted,Pre-Op Q EQ VAS,Post-Op Q EQ VAS,Hip Replacement EQ VAS Post-Op Q Predicted,Hip Replacement Pre-Op Q Pain,Hip Replacement Pre-Op Q Sudden Pain,Hip Replacement Pre-Op Q Night Pain,Hip Replacement Pre-Op Q Washing,Hip Replacement Pre-Op Q Transport,Hip Replacement Pre-Op Q Dressing,Hip Replacement Pre-Op Q Shopping,Hip Replacement Pre-Op Q Walking,Hip Replacement Pre-Op Q Limping,Hip Replacement Pre-Op Q Stairs,Hip Replacement Pre-Op Q Standing,Hip Replacement Pre-Op Q Work,Hip Replacement Pre-Op Q Score,Hip Replacement Post-Op Q Pain,Hip Replacement Post-Op Q Sudden Pain,Hip Replacement Post-Op Q Night Pain,Hip Replacement Post-Op Q Washing,Hip Replacement Post-Op Q Transport,Hip Replacement Post-Op Q Dressing,Hip Replacement Post-Op Q Shopping,Hip Replacement Post-Op Q Walking,Hip Replacement Post-Op Q Limping,Hip Replacement Post-Op Q Stairs,Hip Replacement Post-Op Q Standing,Hip Replacement Post-Op Q Work,Hip Replacement Post-Op Q Score,Hip Replacement OHS Post-Op Q Predicted,CSVYear
cat,cat,u8,cat,cat,f32,u8,i32,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u32,f32,u8,i32,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u32,f32,f32,u16,u16,cat,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,f32,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,f32,cat,cat
"""ADP02""","""Hip Replacement""",0,"""2016/17""",null,null,2,1,2,1,1,1,0,0,0,0,0,0,1,0,0,0,0,1,null,null,null,null,null,99999,null,2,0,2,2,1,1,2,1,1,2,1,2,2,2,2,2,2,11211,0.88,null,null,80,null,0,2,1,3,1,2,3,2,1,0,2,2,19.00,3,4,2,4,3,2,4,4,3,2,2,3,36.00,"""3.777.303.196""","""2016/17"""
"""ADP02""","""Hip Replacement""",0,"""2016/17""",null,null,2,1,3,2,1,1,0,0,0,0,0,0,0,0,0,0,0,0,2,2,2,3,1,22231,0.05,2,0,1,2,2,1,1,1,1,1,1,2,2,2,2,2,2,21111,0.85,0.67,30,70,"""6.592.244.892""",0,1,0,2,2,1,1,1,0,2,2,1,13.00,4,4,4,4,3,3,4,4,4,4,4,4,46.00,"""3.558.681.573""","""2016/17"""
"""ADP02""","""Hip Replacement""",0,"""2016/17""",null,null,1,1,2,2,1,1,0,1,0,0,0,0,0,0,0,0,0,1,2,2,3,3,2,22332,-0.07,2,0,1,1,2,1,2,1,1,2,1,2,2,2,2,2,2,21211,0.81,0.70,40,60,"""6.810.332.786""",1,0,0,1,1,1,0,1,0,1,1,0,7.00,1,4,3,3,2,3,2,4,1,2,4,2,31.00,"""3.293.405.433""","""2016/17"""
"""ADP02""","""Hip Replacement""",0,"""2016/17""",null,null,2,1,4,2,1,2,0,0,0,0,0,0,0,1,0,0,0,0,2,2,2,3,1,22231,0.05,2,0,1,2,2,1,2,2,2,3,2,2,2,2,1,2,1,21222,0.62,0.71,50,55,"""6.883.315.758""",0,4,0,2,2,1,3,2,1,2,2,2,21.00,0,0,0,3,2,2,2,1,1,2,2,0,15.00,"""3.867.962.866""","""2016/17"""
"""ADP02""","""Hip Replacement""",0,"""2016/17""",null,null,2,1,4,2,1,2,0,0,0,0,0,0,0,0,0,0,0,0,2,2,2,3,2,22232,-0.02,2,0,1,2,1,1,1,1,1,1,1,2,2,2,2,2,2,11111,1.00,0.74,20,90,"""7.226.409.643""",0,0,0,1,1,1,2,2,0,2,1,0,10.00,4,4,3,4,4,4,4,4,4,4,4,4,47.00,"""3.686.746.026""","""2016/17"""
"""ADP02""","""Hip Replacement""",0,"""2016/17""",null,null,2,1,2,2,1,1,0,1,0,0,0,0,0,0,0,0,0,1,2,1,2,3,2,21232,0.09,2,0,1,2,1,1,1,1,1,1,1,2,2,2,2,2,2,11111,1.00,0.75,80,100,"""7.907.610.235""",0,1,1,4,2,3,3,3,1,3,3,1,25.00,3,4,4,4,4,4,4,4,4,4,4,3,46.00,"""4.162.250.126""","""2016/17"""
"""ADP02""","""Hip Replacement""",0,"""2016/17""",null,null,2,1,1,2,1,1,0,1,0,0,0,0,0,0,0,0,0,

In [18]:
# Analyze missing values by Age Band
missing_by_age = (
    df.group_by('Age Band')
    .agg([pl.col(c).is_null().sum().alias(c) for c in df.columns if c != 'Age Band'])
    .sort('Age Band')
)
print("Missing values by Age Band:")
print(missing_by_age)

Missing values by Age Band:
shape: (9, 82)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ Age ┆ Pro ┆ Pro ┆ Rev ┆ Yea ┆ Gen ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Hea ┆ Hig ┆ Str ┆ Cir ┆ Lun ┆ Dia ┆ Kid ┆ Ner ┆ Liv ┆ Can ┆ Dep ┆ Art ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Hip ┆ Pre ┆ Pos ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆

The following columns are defined in the dictionary with 1 = Yes and 9 = Missing. In these binary indicator fields, it is common data science practice to treat a missing value (9) as a "No" (0), as it implies the patient did not report the condition [Conversation History].
• Arthritis Indicator (ARTHRITIS)
• Cancer Indicator (CANCER)
• Circulation Indicator (CIRCULATION)
• Depression Indicator (DEPRESSION)
• Diabetes Indicator (DIABETES)
• Heart Disease Indicator (HEART_DISEASE)
• High Blood Pressure Indicator (HIGH_BP)
• Kidney Disease Indicator (KIDNEY_DISEASE)
• Liver Disease Indicator (LIVER_DISEASE)
• Lung Disease Indicator (LUNG_DISEASE)
• Nervous System Indicator (NERVOUS_SYSTEM)
• Stroke Indicator (STROKE)


In [ ]:
# Replace missing values (originally 9) with 0 in binary indicator columns
# Already handlled by previous step, but we can verify that all 9s have been replaced
indicator_columns = [
    'Arthritis', 'Cancer', 'Circulation', 'Depression', 'Diabetes',
    'Heart Disease', 'High Bp', 'Kidney Disease', 'Liver Disease',
    'Lung Disease', 'Nervous System', 'Stroke'
]

# Check which indicator columns exist and have null values
existing_cols = [c for c in indicator_columns if c in df.columns]
missing_cols  = [c for c in indicator_columns if c not in df.columns]

if missing_cols:
    print(f"Columns NOT found in dataframe: {missing_cols}")

null_check = df.select(existing_cols).null_count()
print("Null counts for indicator columns:")
print(null_check)


Null counts for indicator columns:
shape: (1, 12)
┌────────┬────────┬────────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┐
│ Arthri ┆ Cancer ┆ Circul ┆ Depre ┆ Diabe ┆ Heart ┆ High  ┆ Kidne ┆ Liver ┆ Lung  ┆ Nervo ┆ Strok │
│ tis    ┆ ---    ┆ ation  ┆ ssion ┆ tes   ┆ Disea ┆ Bp    ┆ y Dis ┆ Disea ┆ Disea ┆ us    ┆ e     │
│ ---    ┆ u32    ┆ ---    ┆ ---   ┆ ---   ┆ se    ┆ ---   ┆ ease  ┆ se    ┆ se    ┆ Syste ┆ ---   │
│ u32    ┆        ┆ u32    ┆ u32   ┆ u32   ┆ ---   ┆ u32   ┆ ---   ┆ ---   ┆ ---   ┆ m     ┆ u32   │
│        ┆        ┆        ┆       ┆       ┆ u32   ┆       ┆ u32   ┆ u32   ┆ u32   ┆ ---   ┆       │
│        ┆        ┆        ┆       ┆       ┆       ┆       ┆       ┆       ┆       ┆ u32   ┆       │
╞════════╪════════╪════════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╡
│ 0      ┆ 0      ┆ 0      ┆ 0     ┆ 0     ┆ 0     ┆ 0     ┆ 0     ┆ 0     ┆ 0     ┆ 0     ┆ 0     │
└────────┴────────┴────────┴───────┴─────

Delete patient records where they havent responded for 12 questions related to oxford hop score 

these 12 columns represent the individual dimensions of the Oxford Hip Score (OHS), which is the primary disease-specific measure for hip replacements

We removed observations with missing values" to ensure that the models were trained on fully informative and structured data. By removing rows where these functional questions were skipped, you ensure your model follows a robust real-world benchmark

The sources identify that preoperative Q score dimensions are among the most important predictors for surgical success. Specifically, the question regarding limping (HR_Q1_LIMPING) is cited as a top predictor for postoperative outcomes. Imputing values for such high-impact variables would introduce significant noise and bias, potentially causing the model to misidentify "Non-Responders"

Hip Replacement Pre-Op Q Pain                      0.15
Hip Replacement Pre-Op Q Sudden Pain               0.89
Hip Replacement Pre-Op Q Night Pain                0.86
Hip Replacement Pre-Op Q Washing                   0.11
Hip Replacement Pre-Op Q Transport                 0.90
Hip Replacement Pre-Op Q Dressing                  0.90
Hip Replacement Pre-Op Q Shopping                  0.91
Hip Replacement Pre-Op Q Walking                   0.94
Hip Replacement Pre-Op Q Limping                   0.89
Hip Replacement Pre-Op Q Stairs                    0.93
Hip Replacement Pre-Op Q Standing                  0.91
Hip Replacement Pre-Op Q Work                      0.90

In [20]:
# Remove rows where any of the 12 Hip Replacement Pre-Op Q dimensions have missing values
hr_pre_cols = [col for col in df.columns if col.startswith('Hip Replacement Pre-Op Q') and col != 'Hip Replacement Pre-Op Q Score']

print("Hip Replacement Pre-Op Q columns (excluding Score):", hr_pre_cols)
print("Number of columns:", len(hr_pre_cols))

initial_shape = df.shape
df = df.drop_nulls(subset=hr_pre_cols)
final_shape = df.shape

print(f"Removed {initial_shape[0] - final_shape[0]} rows with missing values in Hip Replacement Pre-Op Q dimensions.")
print(f"New dataframe shape: {final_shape}")

Hip Replacement Pre-Op Q columns (excluding Score): ['Hip Replacement Pre-Op Q Pain', 'Hip Replacement Pre-Op Q Sudden Pain', 'Hip Replacement Pre-Op Q Night Pain', 'Hip Replacement Pre-Op Q Washing', 'Hip Replacement Pre-Op Q Transport', 'Hip Replacement Pre-Op Q Dressing', 'Hip Replacement Pre-Op Q Shopping', 'Hip Replacement Pre-Op Q Walking', 'Hip Replacement Pre-Op Q Limping', 'Hip Replacement Pre-Op Q Stairs', 'Hip Replacement Pre-Op Q Standing', 'Hip Replacement Pre-Op Q Work']
Number of columns: 12
Removed 1417 rows with missing values in Hip Replacement Pre-Op Q dimensions.
New dataframe shape: (123427, 82)



Missing values for Hip Replacement Pre-Op Q Score t0_HR_Score and Hip Replacement Post-Op Q Score t1_HR_Score can be calaculated by adding up the 12 questions
Hip ReplacementScore = It is the sum total of the Oxford Hip Score (OHS), a validated 12-question survey used by the NHS to establish a baseline for patient health

Missing values for Pre-Op Q EQ5D Index Profile can be calculated by concatenating Pre-Op Q Mobility	Pre-Op Q Self-Care	Pre-Op Q Activity	Pre-Op Q Discomfort	Pre-Op Q Anxiety as string

Missing value for post-Op Q EQ5D Index Profile can be calcualted by concatinating Post-Op Q Mobility	Post-Op Q Self-Care	Post-Op Q Activity	Post-Op Q Discomfort	Post-Op Q Anxiety


In [21]:
# Calculate missing values for Hip Replacement Pre-Op Q Score by summing the 12 dimension columns
t0_hr_cols = [col for col in df.columns if col.startswith('Hip Replacement Pre-Op Q') and col != 'Hip Replacement Pre-Op Q Score']

print("Hip Replacement Pre-Op Q columns (excluding Score):", t0_hr_cols)
print("Number of Hip Replacement Pre-Op Q columns:", len(t0_hr_cols))

if len(t0_hr_cols) == 12:
    df = df.with_columns(
        pl.when(pl.col('Hip Replacement Pre-Op Q Score').is_null())
          .then(pl.sum_horizontal([pl.col(c) for c in t0_hr_cols]))
          .otherwise(pl.col('Hip Replacement Pre-Op Q Score'))
          .alias('Hip Replacement Pre-Op Q Score')
    )
    print("Missing values in Hip Replacement Pre-Op Q Score filled by summing the 12 Hip Replacement Pre-Op Q columns.")
else:
    print(f"Warning: Expected 12 Hip Replacement Pre-Op Q columns, but found {len(t0_hr_cols)}.")

Hip Replacement Pre-Op Q columns (excluding Score): ['Hip Replacement Pre-Op Q Pain', 'Hip Replacement Pre-Op Q Sudden Pain', 'Hip Replacement Pre-Op Q Night Pain', 'Hip Replacement Pre-Op Q Washing', 'Hip Replacement Pre-Op Q Transport', 'Hip Replacement Pre-Op Q Dressing', 'Hip Replacement Pre-Op Q Shopping', 'Hip Replacement Pre-Op Q Walking', 'Hip Replacement Pre-Op Q Limping', 'Hip Replacement Pre-Op Q Stairs', 'Hip Replacement Pre-Op Q Standing', 'Hip Replacement Pre-Op Q Work']
Number of Hip Replacement Pre-Op Q columns: 12
Missing values in Hip Replacement Pre-Op Q Score filled by summing the 12 Hip Replacement Pre-Op Q columns.


In [24]:
# Calculate missing values for Hip Replacement Post-Op Q Score by summing the 12 dimension columns
t1_hr_cols = [col for col in df.columns if col.startswith('Hip Replacement Post-Op Q') and col != 'Hip Replacement Post-Op Q Score' and 'Predicted' not in col]

print("Hip Replacement Post-Op Q columns (excluding Score and Predicted):", t1_hr_cols)
print("Number of Hip Replacement Post-Op Q columns:", len(t1_hr_cols))

# Check for nulls in these columns before summing
null_check = df.select(t1_hr_cols).null_count()
print("\nNull counts in Post-Op Q dimension columns:")
print(null_check)

if len(t1_hr_cols) == 12:
    # True for any row where at least one component is null
    has_null = pl.any_horizontal([pl.col(c).is_null() for c in t1_hr_cols])

    df = df.with_columns(
        pl.when(has_null)
          # Any null component → score must be null (partial responses are invalid)
          .then(None)
          .when(pl.col('Hip Replacement Post-Op Q Score').is_null())
          # Score missing but all components present → calculate it
          .then(pl.sum_horizontal([pl.col(c) for c in t1_hr_cols]))
          .otherwise(pl.col('Hip Replacement Post-Op Q Score'))
          .alias('Hip Replacement Post-Op Q Score')
    )
    print("\nPost-Op Q Score: set to null where any component is null; filled by sum where score was missing.")
    print(f"Remaining nulls in Post-Op Q Score: {df['Hip Replacement Post-Op Q Score'].null_count()}")
else:
    print(f"Warning: Expected 12 Hip Replacement Post-Op Q columns, but found {len(t1_hr_cols)}.")

Hip Replacement Post-Op Q columns (excluding Score and Predicted): ['Hip Replacement Post-Op Q Pain', 'Hip Replacement Post-Op Q Sudden Pain', 'Hip Replacement Post-Op Q Night Pain', 'Hip Replacement Post-Op Q Washing', 'Hip Replacement Post-Op Q Transport', 'Hip Replacement Post-Op Q Dressing', 'Hip Replacement Post-Op Q Shopping', 'Hip Replacement Post-Op Q Walking', 'Hip Replacement Post-Op Q Limping', 'Hip Replacement Post-Op Q Stairs', 'Hip Replacement Post-Op Q Standing', 'Hip Replacement Post-Op Q Work']
Number of Hip Replacement Post-Op Q columns: 12

Null counts in Post-Op Q dimension columns:
shape: (1, 12)
┌────────┬────────┬────────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┐
│ Hip    ┆ Hip    ┆ Hip    ┆ Hip   ┆ Hip   ┆ Hip   ┆ Hip   ┆ Hip   ┆ Hip   ┆ Hip   ┆ Hip   ┆ Hip   │
│ Replac ┆ Replac ┆ Replac ┆ Repla ┆ Repla ┆ Repla ┆ Repla ┆ Repla ┆ Repla ┆ Repla ┆ Repla ┆ Repla │
│ ement  ┆ ement  ┆ ement  ┆ cemen ┆ cemen ┆ cemen ┆ cemen ┆ cemen ┆ cem

 Remove Patient that has not responded to respective questions - 

Pre-Op Q Mobility                                  3.32
Pre-Op Q Self-Care                                 3.38
Pre-Op Q Activity                                  3.44
Pre-Op Q Discomfort                                4.16
Pre-Op Q Anxiety                                   3.88

Post-Op Q Mobility                                 2.11
Post-Op Q Self-Care                                1.92
Post-Op Q Activity                                 2.14
Post-Op Q Discomfort                               2.76
Post-Op Q Anxiety                                  2.18
 
 Loss of Clinical Precision: These dimensions are used to identify "Non-Responders". If you impute a "1" (No Problems) for a patient who actually had severe problems, you introduce noise that can lead the model to wrongly predict that a patient will not reach the Minimal Important Difference (MID).
• Impact on Calculated Index: The EQ-5D Index is calculated by subtracting specific weights from a 1.0 baseline based on these five digits. An imputed value would create a "hallucinated" index score, leading to selection bias.
• Invalid Value Errors: As established in our conversation history, you cannot replace these with 0. The valid range for these dimensions is 1 to 3 (1=No problems, 2=Moderate, 3=Extreme)

In [25]:
# Remove patients that have not responded to EQ-5D questions to avoid noise
eq5d_cols = [
    'Pre-Op Q Mobility', 'Pre-Op Q Self-Care', 'Pre-Op Q Activity', 'Pre-Op Q Discomfort', 'Pre-Op Q Anxiety',
    'Post-Op Q Mobility', 'Post-Op Q Self-Care', 'Post-Op Q Activity', 'Post-Op Q Discomfort', 'Post-Op Q Anxiety'
]

initial_shape = df.shape
df = df.drop_nulls(subset=eq5d_cols)
final_shape = df.shape

print(f"Removed {initial_shape[0] - final_shape[0]} rows with missing values in EQ-5D dimensions.")
print(f"New dataframe shape: {final_shape}")

Removed 11608 rows with missing values in EQ-5D dimensions.
New dataframe shape: (111819, 82)


• Pre-Op Q Assisted (1.08% missing):
    ◦ Dictionary: 1 = Yes, 2 = No, 9 = Missing.
    ◦ Strategy: Since the missingness is tiny, perform Mode Imputation to "2" (No), which is the dominant value in the provided CSV excerpts.

In [28]:
# Mode imputation for Pre-Op Q Assisted — fill missing with 2 (No)
df = df.with_columns(pl.col('Pre-Op Q Assisted').fill_null(2))
print("Missing values in 'Pre-Op Q Assisted' filled with 2 (No).")

Missing values in 'Pre-Op Q Assisted' filled with 2 (No).


Pre-Op Q Symptom Period (0.88% missing):
    ◦ Dictionary: 1 = <1 yr, 2 = 1–5 yrs, 3 = 6–10 yrs, 4 = >10 yrs, 9 = Missing.
    ◦ Strategy: The sources identify Symptom Duration as a strong predictor for postoperative outcomes. Because this is a high-impact variable, do not use mode imputation. Listwise deletion for these specific rows is recommended to avoid introducing noise into a critical predictor

In [ ]:
# Listwise deletion for Pre-Op Q Symptom Period
initial_shape = df.shape
df = df.drop_nulls(subset=['Pre-Op Q Symptom Period'])
final_shape = df.shape

print(f"Removed {initial_shape[0] - final_shape[0]} rows with missing 'Pre-Op Q Symptom Period'.")
print(f"New dataframe shape: {final_shape}")

Removed 696 rows with missing 'Pre-Op Q Symptom Period'.
New dataframe shape: (111123, 82)


Pre-Op Q Previous Surgery (0.73% missing):
    ◦ Dictionary: 1 = Yes, 2 = No, 9 = Missing.
    ◦ Strategy: Given the extremely low missingness, Mode Imputation to "2" (No) is a safe way to preserve the other data points in those rows

In [29]:
# Mode imputation for Pre-Op Q Previous Surgery — fill missing with 2 (No)
df = df.with_columns(pl.col('Pre-Op Q Previous Surgery').fill_null(2))
print("Missing values in 'Pre-Op Q Previous Surgery' filled with 2 (No).")

Missing values in 'Pre-Op Q Previous Surgery' filled with 2 (No).


Pre-Op Q Living Arrangements (1.35% missing):
    ◦ Dictionary: 1 = With family/partner, 2 = Alone, 3 = Nursing home, 4 = Other, 9 = Missing.
    ◦ Strategy: Keep "9" as a distinct category ("Unknown/Not Specified"). In predictive modelling, a patient's choice to skip a question about their home life can sometimes be an informative feature regarding their social support system

In [30]:
# Replace null with 9 to keep Unknown/Not Specified as a distinct category
df = df.with_columns(pl.col('Pre-Op Q Living Arrangements').fill_null(9))
print("Missing values in 'Pre-Op Q Living Arrangements' filled with 9 (Unknown/Not Specified).")

Missing values in 'Pre-Op Q Living Arrangements' filled with 9 (Unknown/Not Specified).


Pre-Op Q Disability (5.88% missing):
Dictionary: 1 = Yes, 2 = No, 9 = Missing.
    ◦ Strategy: This is your highest missingness. Do not replace 9 with 0, as the dictionary defines "2" as the code for "No".
    ◦ Best approach: Treat "9" as a third categorical level ("Not Disclosed"). Patients may hesitate to self-identify as "disabled," and forcing them into "Yes" or "No" would introduce selection bias

In [31]:
# Replace null with 9 to specify Not Disclosed as a distinct category
df = df.with_columns(pl.col('Pre-Op Q Disability').fill_null(9))
print("Missing values in 'Pre-Op Q Disability' filled with 9 (Not Disclosed).")

Missing values in 'Pre-Op Q Disability' filled with 9 (Not Disclosed).


In [32]:
# Function to calculate EQ-5D-5L index from profile using UK value set
def calculate_eq5d_index(profile):
    if profile is None or len(str(profile)) != 5:
        return None
    try:
        digits = [int(d) for d in str(profile)]
        if any(d not in [1, 2, 3] for d in digits):
            return None  # Invalid or missing
    except (ValueError, TypeError):
        return None

    # UK value set weights
    constant   = 0.081
    n3_penalty = 0.269

    dim_weights = {
        0: {'2': 0.069, '3': 0.314},  # Mobility
        1: {'2': 0.104, '3': 0.214},  # Self-care
        2: {'2': 0.036, '3': 0.094},  # Usual activities
        3: {'2': 0.123, '3': 0.386},  # Pain/discomfort
        4: {'2': 0.071, '3': 0.236}   # Anxiety/depression
    }

    total_deduction = constant
    has_level_3 = False

    for i, level in enumerate(digits):
        if level == 2:
            total_deduction += dim_weights[i]['2']
        elif level == 3:
            total_deduction += dim_weights[i]['3']
            has_level_3 = True

    if has_level_3:
        total_deduction += n3_penalty

    return 1.0 - total_deduction

# Verify the EQ-5D index calculation for existing values
df_verify = df.filter(pl.col('Pre-Op Q EQ5D Index').is_not_null())
df_verify = df_verify.with_columns(
    pl.col('Pre-Op Q EQ5D Index Profile')
      .map_elements(calculate_eq5d_index, return_dtype=pl.Float64)
      .alias('calculated_index')
)
df_verify = df_verify.with_columns(
    (pl.col('Pre-Op Q EQ5D Index') - pl.col('calculated_index')).alias('difference')
)

print("Verification of EQ-5D index calculation:")
print(f"Number of non-missing Pre-Op Q EQ5D Index: {df_verify.shape[0]}")
print(f"Mean difference: {df_verify['difference'].mean():.6f}")
print(f"Max difference:  {df_verify['difference'].max():.6f}")
print(f"Min difference:  {df_verify['difference'].min():.6f}")
print(f"Number of exact matches (diff < 1e-6): {(df_verify['difference'].abs() < 1e-6).sum()}")

print("\nFirst 10 examples:")
print(df_verify.select(['Pre-Op Q EQ5D Index Profile', 'Pre-Op Q EQ5D Index', 'calculated_index', 'difference']).head(10))

Verification of EQ-5D index calculation:
Number of non-missing Pre-Op Q EQ5D Index: 111123
Mean difference: 0.000305
Max difference:  0.081000
Min difference:  -0.000000
Number of exact matches (diff < 1e-6): 110704

First 10 examples:
shape: (10, 4)
┌─────────────────────────────┬─────────────────────┬──────────────────┬────────────┐
│ Pre-Op Q EQ5D Index Profile ┆ Pre-Op Q EQ5D Index ┆ calculated_index ┆ difference │
│ ---                         ┆ ---                 ┆ ---              ┆ ---        │
│ u32                         ┆ f32                 ┆ f64              ┆ f64        │
╞═════════════════════════════╪═════════════════════╪══════════════════╪════════════╡
│ 22231                       ┆ 0.05                ┆ 0.06             ┆ -0.00      │
│ 22332                       ┆ -0.07               ┆ -0.07            ┆ -0.00      │
│ 22231                       ┆ 0.05                ┆ 0.06             ┆ -0.00      │
│ 22232                       ┆ -0.02               ┆ -0.02  

The verification output shows that the EQ-5D index calculation formula is correct for the vast majority of cases. Out of 117,824 non-missing t0_EQ5D_Index values:

117,354 (99.6%) are exact matches (difference < 1e-6, likely due to floating-point precision).
The mean difference is negligible (0.00032), and the maximum difference is 0.081, which could be due to rounding in the original data or slight variations in value sets.

In [33]:
# Fill missing Pre-Op Q EQ5D Index Profile by concatenating the 5 EQ5D component columns
eq5d_pre_cols = ['Pre-Op Q Mobility', 'Pre-Op Q Self-Care', 'Pre-Op Q Activity', 'Pre-Op Q Discomfort', 'Pre-Op Q Anxiety']

df = df.with_columns(
    pl.when(pl.col('Pre-Op Q EQ5D Index Profile').is_null())
      .then(pl.concat_str([pl.col(c).cast(pl.Utf8) for c in eq5d_pre_cols], separator=''))
      .otherwise(pl.col('Pre-Op Q EQ5D Index Profile'))
      .alias('Pre-Op Q EQ5D Index Profile')
)
print("Missing values in Pre-Op Q EQ5D Index Profile filled by concatenating the 5 EQ5D components.")

df = df.with_columns(
    pl.when(pl.col('Pre-Op Q EQ5D Index').is_null())
      .then(
          pl.col('Pre-Op Q EQ5D Index Profile')
            .map_elements(calculate_eq5d_index, return_dtype=pl.Float64)
      )
      .otherwise(pl.col('Pre-Op Q EQ5D Index'))
      .alias('Pre-Op Q EQ5D Index')
)
print("Missing values in Pre-Op Q EQ5D Index filled using the calculated index from Pre-Op Q EQ5D Index Profile.")

Missing values in Pre-Op Q EQ5D Index Profile filled by concatenating the 5 EQ5D components.
Missing values in Pre-Op Q EQ5D Index filled using the calculated index from Pre-Op Q EQ5D Index Profile.


In [34]:
# Fill missing Post-Op Q EQ5D Index Profile by concatenating the 5 EQ5D component columns
eq5d_post_cols = ['Post-Op Q Mobility', 'Post-Op Q Self-Care', 'Post-Op Q Activity', 'Post-Op Q Discomfort', 'Post-Op Q Anxiety']

df = df.with_columns(
    pl.when(pl.col('Post-Op Q EQ5D Index Profile').is_null())
      .then(pl.concat_str([pl.col(c).cast(pl.Utf8) for c in eq5d_post_cols], separator=''))
      .otherwise(pl.col('Post-Op Q EQ5D Index Profile'))
      .alias('Post-Op Q EQ5D Index Profile')
)
print("Missing values in Post-Op Q EQ5D Index Profile filled by concatenating the 5 EQ5D components.")

df = df.with_columns(
    pl.when(pl.col('Post-Op Q EQ5D Index').is_null())
      .then(
          pl.col('Post-Op Q EQ5D Index Profile')
            .map_elements(calculate_eq5d_index, return_dtype=pl.Float64)
      )
      .otherwise(pl.col('Post-Op Q EQ5D Index'))
      .alias('Post-Op Q EQ5D Index')
)
print("Missing values in Post-Op Q EQ5D Index filled using the calculated index from Post-Op Q EQ5D Index Profile.")

Missing values in Post-Op Q EQ5D Index Profile filled by concatenating the 5 EQ5D components.
Missing values in Post-Op Q EQ5D Index filled using the calculated index from Post-Op Q EQ5D Index Profile.


Ignore columns -

post operation questions, the colmn with t1 and Q init
Hip Replacement EQ5D Index Post-Op Q Predicted - predicted by liner regression
Hip Replacement OHS Post-Op Q Predicted 
Hip Replacement EQ VAS Post-Op Q Predicted


In [35]:
# Identify columns to remove and keep
all_columns = df.columns  # Polars returns a plain list

# Columns to keep even if they contain 'Post-Op'
keep_post_op = ['Post-Op Q EQ5D Index Profile', 'Post-Op Q EQ5D Index', 'Post-Op Q EQ VAS', 'Post-Op Q HR_Score']

# Remove Post-Op columns (except the specified ones), Predicted columns, and CSVYear
removed_columns = [
    col for col in all_columns
    if ('Post-Op' in col and col not in keep_post_op) or 'Predicted' in col or col == 'CSVYear'
]
kept_columns = [col for col in all_columns if col not in removed_columns]

print("Columns to be removed:")
print(removed_columns)
print(f"\nTotal removed: {len(removed_columns)}")

print("\nColumns to be kept:")
print(kept_columns)
print(f"\nTotal kept: {len(kept_columns)}")

Columns to be removed:
['Post-Op Q Assisted', 'Post-Op Q Assisted By', 'Post-Op Q Living Arrangements', 'Post-Op Q Disability', 'Post-Op Q Mobility', 'Post-Op Q Self-Care', 'Post-Op Q Activity', 'Post-Op Q Discomfort', 'Post-Op Q Anxiety', 'Post-Op Q Satisfaction', 'Post-Op Q Sucess', 'Post-Op Q Allergy', 'Post-Op Q Bleeding', 'Post-Op Q Wound', 'Post-Op Q Urine', 'Post-Op Q Further Surgery', 'Post-Op Q Readmitted', 'Hip Replacement EQ5D Index Post-Op Q Predicted', 'Hip Replacement EQ VAS Post-Op Q Predicted', 'Hip Replacement Post-Op Q Pain', 'Hip Replacement Post-Op Q Sudden Pain', 'Hip Replacement Post-Op Q Night Pain', 'Hip Replacement Post-Op Q Washing', 'Hip Replacement Post-Op Q Transport', 'Hip Replacement Post-Op Q Dressing', 'Hip Replacement Post-Op Q Shopping', 'Hip Replacement Post-Op Q Walking', 'Hip Replacement Post-Op Q Limping', 'Hip Replacement Post-Op Q Stairs', 'Hip Replacement Post-Op Q Standing', 'Hip Replacement Post-Op Q Work', 'Hip Replacement Post-Op Q Score', 

In [36]:
# Create new dataframe with selected columns
df_selected = df.select(kept_columns)

print("New dataframe created with selected columns.")
print("Shape of new dataframe:", df_selected.shape)
print("Columns in new dataframe:")
print(df_selected.columns)

New dataframe created with selected columns.
Shape of new dataframe: (111123, 48)
Columns in new dataframe:
['Provider Code', 'Procedure', 'Revision Flag', 'Year', 'Age Band', 'Gender', 'Pre-Op Q Assisted', 'Pre-Op Q Assisted By', 'Pre-Op Q Symptom Period', 'Pre-Op Q Previous Surgery', 'Pre-Op Q Living Arrangements', 'Pre-Op Q Disability', 'Heart Disease', 'High Bp', 'Stroke', 'Circulation', 'Lung Disease', 'Diabetes', 'Kidney Disease', 'Nervous System', 'Liver Disease', 'Cancer', 'Depression', 'Arthritis', 'Pre-Op Q Mobility', 'Pre-Op Q Self-Care', 'Pre-Op Q Activity', 'Pre-Op Q Discomfort', 'Pre-Op Q Anxiety', 'Pre-Op Q EQ5D Index Profile', 'Pre-Op Q EQ5D Index', 'Post-Op Q EQ5D Index Profile', 'Post-Op Q EQ5D Index', 'Pre-Op Q EQ VAS', 'Post-Op Q EQ VAS', 'Hip Replacement Pre-Op Q Pain', 'Hip Replacement Pre-Op Q Sudden Pain', 'Hip Replacement Pre-Op Q Night Pain', 'Hip Replacement Pre-Op Q Washing', 'Hip Replacement Pre-Op Q Transport', 'Hip Replacement Pre-Op Q Dressing', 'Hip Rep

In [37]:
# List columns with missing values in df_selected
null_counts  = df_selected.null_count()
missing_cols = [c for c in df_selected.columns if null_counts[c][0] > 0]

print("Columns with missing values in df_selected:")
print(missing_cols)
print(f"\nTotal columns with missing values: {len(missing_cols)}")

print("\nMissing values count per column:")
print(null_counts.select(missing_cols))

Columns with missing values in df_selected:
['Age Band', 'Gender', 'Pre-Op Q EQ VAS', 'Post-Op Q EQ VAS']

Total columns with missing values: 4

Missing values count per column:
shape: (1, 4)
┌──────────┬────────┬─────────────────┬──────────────────┐
│ Age Band ┆ Gender ┆ Pre-Op Q EQ VAS ┆ Post-Op Q EQ VAS │
│ ---      ┆ ---    ┆ ---             ┆ ---              │
│ u32      ┆ u32    ┆ u32             ┆ u32              │
╞══════════╪════════╪═════════════════╪══════════════════╡
│ 8686     ┆ 8686   ┆ 6372            ┆ 3435             │
└──────────┴────────┴─────────────────┴──────────────────┘


In [40]:
# Listwise deletion for Age Band and Gender — both are critical casemix variables
initial_shape = df_selected.shape
df_final = df_selected.drop_nulls(subset=['Age Band', 'Gender'])
final_shape = df_final.shape

print(f"Removed {initial_shape[0] - final_shape[0]} rows with missing Age Band or Gender.")
print(f"New dataframe shape: {final_shape}")

# Recompute missing counts from df_final
null_counts_final = df_final.null_count()
missing_cols_final = [c for c in df_final.columns if null_counts_final[c][0] > 0]

print(f"\nColumns with missing values in df_final: {missing_cols_final}")
print(f"Total columns with missing values: {len(missing_cols_final)}")
if missing_cols_final:
    print("\nMissing values count per column:")
    print(null_counts_final.select(missing_cols_final))

Removed 8686 rows with missing Age Band or Gender.
New dataframe shape: (102437, 48)

Columns with missing values in df_final: ['Pre-Op Q EQ VAS', 'Post-Op Q EQ VAS']
Total columns with missing values: 2

Missing values count per column:
shape: (1, 2)
┌─────────────────┬──────────────────┐
│ Pre-Op Q EQ VAS ┆ Post-Op Q EQ VAS │
│ ---             ┆ ---              │
│ u32             ┆ u32              │
╞═════════════════╪══════════════════╡
│ 5947            ┆ 3209             │
└─────────────────┴──────────────────┘


In [ ]:
# Drop EQ VAS columns — not used in the model
df_final = df_final.drop(['Pre-Op Q EQ VAS', 'Post-Op Q EQ VAS'])

print(f"Dropped 'Pre-Op Q EQ VAS' and 'Post-Op Q EQ VAS'.")
print(f"Final dataframe shape: {df_final.shape}")

# Recompute missing counts from df_final
null_counts_final = df_final.null_count()
missing_cols_final = [c for c in df_final.columns if null_counts_final[c][0] > 0]

print(f"\nColumns with missing values in df_final: {missing_cols_final}")
print(f"Total columns with missing values: {len(missing_cols_final)}")
if missing_cols_final:
    print("\nMissing values count per column:")
    print(null_counts_final.select(missing_cols_final))


Columns with missing values in df_final: []
Total columns with missing values: 0


In [46]:
# Save the cleaned dataframe to a Parquet file
final_shape = df_final.shape
df_final.write_parquet(f'./data/interim/2.1-Hip-data-preparation-Manual.parquet', compression='gzip')
print(f"Dataframe saved to './data/interim/2.1-Hip-data-preparation-Manual.parquet' with shape {final_shape}")

Dataframe saved to './data/interim/2.1-Hip-data-preparation-Manual.parquet' with shape (102437, 46)


for age band and Gender =

 Listwise Deletion (Removal)
The most robust approach for your machine learning model is to remove these 8,686 observations entirely.
• Methodological Alignment: In the research paper evaluating hip replacements, the researchers explicitly stated: "We removed observations with missing values" to ensure the classifiers were trained on fully informative data.
• Statistical Impact: Deleting 8,686 rows from a total population of 130,945 observations represents a loss of only approximately 6.6% of your data. This is a minor trade-off to ensure your model doesn't attempt to learn patterns from "masked" demographics.
• Avoiding Noise: Since age and gender are important casemix-adjustment variables used by the NHS to predict outcomes, attempting to impute them would introduce significant noise

Confusion Matrix - Find out True Negative

Precison - false Positive

Auroc wirh Precsion and recall curve and select a point.

correlation in EQVAS and OHR
what is percentage of improvement of OHR consider as health gain

Startegy for OHS score -

check if there is an increament from past score + 8 / 6 ( recent studies says 6 )
check if the new score is reaching minimum threashold 26
if yes - then its a success for us

else failure for us.